# 📌 Cache-Augmented Generation (CAG)

![Topic](https://img.shields.io/badge/Topic-CAG-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-RAG-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-April%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — **Cache-Augmented Generation (CAG)** is an alternative to RAG that skips real-time retrieval entirely.<br> Instead of searching a database every time a user asks a question, CAG preloads all relevant resources into the model's context and caches its runtime parameters. The preloaded KV-cache enables the model to generate responses directly, eliminating the need for retrieval. <br> To summarize, you can see RAG like a researcher who looks up information in real time for each question and CAG like a person who reads everything relevant the night before and answers from memory. </span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install transformers torch bitsandbytes accelerate` |
| Concepts | RAG, KV Cache |

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

### 1.1 What is CAG?

**CAG** is an architectural framework for LLMs that **preloads a specific knowledge base directly into the model's context window and stores the resulting computational states in a reusable cache.** It takes advantage of the fact that modern LLMs now have massive context windows (100K+ tokens), meaning you can often just load all your documents into the model ahead of time.The key mechanism of the CAG is the KV cache. When a model processes text, it generates KV pairs for every token to calculate relationships. CAG performs this calculation for the entire knowledge base once and saves these states. When a user then asks a question, the model only needs to process the new query tokens, reusing all the precomputed states instantly.

### 1.2 How does it differ from RAG?




**RAG follows a multi-step pipeline for every query** (embed the question → search a vector database → retrieve relevant chunks → feed them to the LLM → generate an answer), introducing complexity,risk of suffer from retrieval errors (pulling the wrong documents) and latency. <br>
**CAG simplifies this** by removing the need for retrieval pipelines, reducing overall system complexity.

### 1.3 When should you use CAG?

**CAG** is ideal when:<br>
<br> 1. **The knowledge base can fit within the model's extended context**
<br> 2. **The latency is critical and real-time retrieval introduces unacceptable delays**
<br> 3. **Simplicity is a priority**

<br>Typical use cases include company policy Q&A, medical guideline lookup, legal document analysis, and internal knowledge bases that don't change frequently.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

**CAG does all the heavy lifting ahead of time during an offline phase.** 
<br> After preloading documents once, the model only needs to process the new query tokens during subsequent requests, making responses significantly faster.

![CAG](../assets/CAG.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Speed** | No retrieval step at all |
| 🟢 | 	**Simplicity** | No vector database, no embedding model, no retrieval pipeline. Fewer moving parts means less to maintain, debug, and monitor in production. |
| 🟢 | **Accuracy** | The model sees the entire knowledge base at once, enabling better multi-hop reasoning. No risk of retrieving the wrong chunks or missing relevant context. |
| 🟢 | **Reliability** | Eliminates retrieval errors entirely.|
| 🔴 | **Context window limit** | Your entire knowledge base must fit in the model's context window.|
| 🔴 | **Stale data** | If your knowledge changes frequently, you must rebuild the KV cache each time.|
| 🔴 | **Upfront compute** | Precomputing and storing the KV cache requires significant initial processing time and memory, especially for large document sets near the context limit. |
| 🔴 | **API access** |Direct KV cache manipulation requires open-source models (like Llama). Most commercial APIs (GPT-4, Claude) don't expose the KV cache, limiting where you can deploy true CAG. |

---
## 4. Code Example

> **Goal:** This script shows the preload of the documents and the save into the kv cahce and the use of saved cache to answer questions.


In [1]:
"""
Cache-Augmented Generation (CAG) with Llama 3.1 — Beginner Example
====================================================================
This script shows the two phases of CAG using Llama 3.1 8B Instruct,
the same model used in the original "Don't Do RAG" research paper.

  Phase 1: Preload documents into the model and save the KV cache.
  Phase 2: Use the saved cache to answer questions instantly (no retrieval).

Requirements:
  pip install torch transformers bitsandbytes accelerate

Hardware:
  - GPU with at least 8 GB VRAM (e.g. T4, RTX 3090) when using 4-bit quantization
  - Without quantization, Llama 3.1 8B needs ~16 GB VRAM

Note: You need a Hugging Face account with access to Llama.
  1. Create an account at https://huggingface.co
  2. Accept the Llama license at https://huggingface.co/meta-llama
  3. Generate a token at https://huggingface.co/settings/tokens
  4. Run: huggingface-cli login
"""

import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from transformers.cache_utils import DynamicCache


# ──────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────

# The model used in the original CAG paper
# Llama 3.1 8B supports a 128K token context window (~100K words)
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Path to save/load the KV cache on disk
CACHE_PATH = "./kv_cache_company_policies.pt"


# ──────────────────────────────────────────────
# STEP 1: LOAD LLAMA 3.1
# ──────────────────────────────────────────────
# We use 4-bit quantization to reduce memory usage.
# Without it: ~16 GB VRAM. With it: ~6 GB VRAM.

print("📦 Loading Llama 3.1 8B (4-bit quantized)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Load weights in 4-bit
    bnb_4bit_use_double_quant=True,        # Double quantization for extra savings
    bnb_4bit_quant_type="nf4",             # NormalFloat4 type (recommended)
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute in bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",  # Automatically distribute across available GPUs
)

print("✅ Model loaded!\n")


# ──────────────────────────────────────────────
# STEP 2: PREPARE THE KNOWLEDGE BASE
# ──────────────────────────────────────────────
# In production this could be company docs, medical guidelines,
# legal texts, etc. Everything must fit within the model's
# context window (128K tokens for Llama 3.1).

knowledge_base = """
== Company Vacation Policy ==
All full-time employees receive 25 days of paid vacation per year.
Vacation days reset on January 1st each year.
Unused vacation days can be carried over up to a maximum of 5 days.
Employees must request vacation at least 2 weeks in advance.
Managers must approve or deny requests within 3 business days.

== Remote Work Policy ==
Employees may work remotely up to 3 days per week.
Remote work requires manager approval and a signed agreement.
All remote workers must be available during core hours: 10 AM - 4 PM.
Equipment stipend: €500 per year for home office setup.

== Expense Policy ==
Business meals are reimbursed up to €50 per person per meal.
Travel expenses require pre-approval for amounts over €200.
Receipts must be submitted within 30 days of the expense.

== IT Security Policy ==
Passwords must be at least 12 characters long.
Two-factor authentication (2FA) is mandatory for all accounts.
Confidential data must never be shared via unencrypted messaging.
Any security incident must be reported to the IT team within 24 hours.
"""


# ──────────────────────────────────────────────
# PHASE 1: BUILD AND SAVE THE KV CACHE
# ──────────────────────────────────────────────
# This is the "offline" step — you do this ONCE.
# The model processes all documents and saves its internal states
# (key-value pairs from every attention layer).

def build_kv_cache(model, tokenizer, knowledge: str) -> DynamicCache:
    """
    Preprocess the knowledge base and return a DynamicCache.

    DynamicCache is Hugging Face's class for storing the key-value
    pairs from all attention layers of the transformer.
    This is the technical core of CAG.
    """
    # Tokenize the knowledge base
    inputs = tokenizer(
        knowledge,
        return_tensors="pt",
    ).to(model.device)

    # Initialize an empty dynamic cache
    past_key_values = DynamicCache()

    # Run a forward pass to compute all KV pairs
    # This is where the model "reads" and "memorizes" all documents
    with torch.no_grad():
        outputs = model(
            **inputs,
            past_key_values=past_key_values,  # Cache gets filled here
            use_cache=True,
        )

    return outputs.past_key_values


def save_cache_to_disk(cache: DynamicCache, path: str):
    """
    Save the KV cache to disk for reuse across sessions.
    You won't need to reprocess the documents again.
    """
    kv_data = []
    for layer_idx in range(len(cache.key_cache)):
        key = cache.key_cache[layer_idx].cpu()
        value = cache.value_cache[layer_idx].cpu()
        kv_data.append((key, value))

    torch.save(kv_data, path)
    print(f"💾 Cache saved to {path}")


def load_cache_from_disk(path: str, device) -> DynamicCache:
    """
    Reload the KV cache from disk.
    No need to reprocess documents!
    """
    kv_data = torch.load(path, map_location=device, weights_only=True)

    cache = DynamicCache()
    for key, value in kv_data:
        cache.key_cache.append(key.to(device))
        cache.value_cache.append(value.to(device))

    print(f"📂 Cache loaded from {path}")
    return cache


# Build the cache (or load it if it already exists on disk)
if os.path.exists(CACHE_PATH):
    print("📂 Existing cache found on disk, loading...")
    kv_cache = load_cache_from_disk(CACHE_PATH, model.device)
else:
    print("⏳ Building KV cache (one-time offline step)...")
    kv_cache = build_kv_cache(model, tokenizer, knowledge_base)

    num_tokens = kv_cache.key_cache[0].shape[2]
    num_layers = len(kv_cache.key_cache)
    print(f"✅ KV cache built!")
    print(f"   → {num_tokens} tokens preloaded")
    print(f"   → {num_layers} attention layers cached")

    # Save for future sessions
    save_cache_to_disk(kv_cache, CACHE_PATH)

# Remember how many tokens belong to the knowledge base
# (needed for cache reset after each query)
NUM_KNOWLEDGE_TOKENS = kv_cache.key_cache[0].shape[2]
print()


# ──────────────────────────────────────────────
# PHASE 2: ANSWER QUERIES USING THE CACHE
# ──────────────────────────────────────────────
# No retrieval, no vector search, no embedding.
# The model already "knows" your documents from the cache.

def clean_up_cache(cache: DynamicCache, num_knowledge_tokens: int):
    """
    Reset the cache to its original state (knowledge base only).

    After each query, the cache accumulates the query's and answer's
    tokens. This function truncates them so the next query starts
    fresh, without interference from previous questions.

    This is the "cache reset" described in the CAG paper.
    """
    for layer_idx in range(len(cache.key_cache)):
        cache.key_cache[layer_idx] = cache.key_cache[layer_idx][
            :, :, :num_knowledge_tokens, :
        ]
        cache.value_cache[layer_idx] = cache.value_cache[layer_idx][
            :, :, :num_knowledge_tokens, :
        ]


def ask_with_cag(
    question: str,
    model,
    tokenizer,
    cache: DynamicCache,
    num_knowledge_tokens: int,
    max_new_tokens: int = 150,
) -> str:
    """
    Answer a question using the preloaded KV cache.

    The model does NOT re-read the documents — it reuses the
    precomputed attention states. Only the question tokens
    are processed. That's what makes CAG so fast.
    """
    # Format the question as a prompt
    prompt = f"\n\nQuestion: {question}\nAnswer:"

    # Tokenize ONLY the new question (not the whole knowledge base again!)
    query_tokens = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    # Generate answer using the cached KV pairs
    # past_key_values=cache → "you already read the docs, here's just the question"
    with torch.no_grad():
        output = model.generate(
            **query_tokens,
            past_key_values=cache,       # ← This is the magic of CAG!
            max_new_tokens=max_new_tokens,
            do_sample=False,             # Deterministic output
            temperature=1.0,
        )

    # Decode only the new tokens (skip the prompt)
    new_tokens = output[0][query_tokens["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Reset the cache for the next question
    clean_up_cache(cache, num_knowledge_tokens)

    return answer.strip()


# ──────────────────────────────────────────────
# TRY IT OUT: Ask questions against your knowledge
# ──────────────────────────────────────────────

questions = [
    "How many vacation days do employees get per year?",
    "How many days per week can I work remotely?",
    "What is the maximum reimbursement for a business meal?",
    "What is the minimum password length?",
    "How long do I have to submit expense receipts?",
]

print("=" * 60)
print("  CAG with Llama 3.1 — Questions & Answers")
print("=" * 60)

for q in questions:
    print(f"\n❓ {q}")

    answer = ask_with_cag(
        question=q,
        model=model,
        tokenizer=tokenizer,
        cache=kv_cache,
        num_knowledge_tokens=NUM_KNOWLEDGE_TOKENS,
    )

    print(f"💬 {answer}")
    print("-" * 60)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 Loading Llama 3.1 8B (4-bit quantized)...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct.
401 Client Error. (Request ID: Root=1-69eb7af6-529773ae37b210f97d5949b4;066c939e-ae5c-42ad-bed1-b70dc1bb30f7)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **CAG replaces retrieval for preloading**
- **Speed gains are dramatic.**
- **Accuracy matches or beats RAG.**
- **CAG works best for stable, bounded knowledge.**

</div>